In [ ]:
!pip install epitran --quiet
import torch
import torch.nn as nn
import torch.nn.functional as F
import epitran
import re
import os


In [ ]:
# raw language files
lang_files = {
    "ar": "data/raw/ar.txt",
    "fi": "data/raw/fi.txt",
    "hu": "data/raw/hu.txt",
    "ru": "data/raw/ru.txt",
}

# Epitran converters
converters = {
    "ar": epitran.Epitran("ara-Arab"),
    "fi": epitran.Epitran("fin-Latn"),
    "hu": epitran.Epitran("hun-Latn"),
    "ru": epitran.Epitran("rus-Cyrl"),
}

def clean_input_text(text):
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\d+", "", text)
    return text.strip()


def convert_to_ipa(file_name, converter):
    ipa_words = []

    with open(file_name, "r", encoding="utf-8") as f:
        for line in f:
            word = clean_input_text(line)

            if not word:
                continue

            try:
                ipa = converter.transliterate(word).strip()

                if ipa:
                    ipa_words.append(ipa)

            except Exception as e:
                print(f"failed: {word} -> {e}")

    return ipa_words


ipa_by_lang = {}

for lang, file_name in lang_files.items():
    ipa_by_lang[lang] = convert_to_ipa(
        file_name,
        converters[lang]
    )

print(ipa_by_lang)



In [ ]:
# Tokenizer
special_tokens = ["<PAD>", "<BOS>", "<EOS>"]
chars = []
for lang, ipa_list in ipa_by_lang.items():
  for ipa in ipa_list:
    chars.extend(ipa)

ipa_list = sorted(list(set(chars)))
print(len(ipa_list))
print(ipa_list)

tokens = special_tokens + ipa_list

id2ipa = {i: token for i, token in enumerate(tokens)}
ipa2id = {token: i for i, token in enumerate(tokens)}

PAD_ID = ipa2id["<PAD>"]
BOS_ID = ipa2id["<BOS>"]
EOS_ID = ipa2id["<EOS>"]

print("vocab:", len(tokens))
print(ipa2id)

In [ ]:
lang2id = {
    "ar": 0,
    "fi": 1,
    "hu": 2,
    "ru": 3,
}
block_size = 24
def encode_word(word):
    ids = [BOS_ID]

    for char in word:
        ids.append(ipa2id[char])

    ids.append(EOS_ID)

    # 最大長を超えたら切る
    ids = ids[:block_size] + [EOS_ID] if len(ids) > block_size + 1 else ids

    # PADで長さをそろえる
    ids += [PAD_ID] * (block_size + 1 - len(ids))

    # 次文字予測なので1個ずらす
    x = torch.tensor(ids[:-1], dtype=torch.long)
    y = torch.tensor(ids[1:], dtype=torch.long)

    return x, y
samples = []

for lang, words in ipa_by_lang.items():
    lang_id = lang2id[lang]

    for word in words:
        x, y = encode_word(word)

        samples.append(
            (x, y, lang_id)
        )

In [ ]:
import random

batch_size = 32

def get_batch():
    batch = random.choices(samples, k=batch_size)

    x = torch.stack([
        sample[0]
        for sample in batch
    ])

    y = torch.stack([
        sample[1]
        for sample in batch
    ])

    lang_ids = torch.tensor([
        sample[2]
        for sample in batch
    ], dtype=torch.long)

    return x, y, lang_ids
xb, yb, lang_ids = get_batch()

print(xb.shape)
print(yb.shape)
print(lang_ids.shape)
print(lang_ids[:10])

In [ ]:
n_embd = 64
vocab_size = len(ipa2id)
num_langs = len(lang2id)

class CharTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # NEW
        self.language_embedding = nn.Embedding(
            num_langs,
            n_embd
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_embd,
            nhead=4,
            dim_feedforward=128,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, idx, lang_ids=None, targets=None, lang_weights=None):
        B, T = idx.shape

        # IPA char embedding
        tok_emb = self.token_embedding_table(idx)

        # position embedding
        pos = torch.arange(
            T,
            dtype=torch.long,
            device=idx.device
        )

        pos_emb = self.position_embedding_table(pos)

        # language embedding
        if lang_weights is None:
              lang_emb = self.language_embedding(lang_ids)

        else:
            lang_emb = (
                lang_weights
                @ self.language_embedding.weight
            )

        # [B, n_embd]
        # ↓
        # [B, 1, n_embd]
        lang_emb = lang_emb[:, None, :]

        # combine
        x = tok_emb + pos_emb + lang_emb

        mask = nn.Transformer.generate_square_subsequent_mask(
            T,
            device=idx.device
        )

        x = self.transformer(
            x,
            mask=mask,
            is_causal=True
        )

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits_flat = logits.reshape(B * T, C)
            targets_flat = targets.reshape(B * T)

            loss = F.cross_entropy(
                logits_flat,
                targets_flat,
                ignore_index=PAD_ID
            )

        return logits, loss
model = CharTransformer()

xb, yb, lang_ids = get_batch()

logits, loss = model(
    xb,
    lang_ids,
    yb
)

print(logits.shape)
print(loss)

In [ ]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

max_iters = 3000
eval_interval = 300

for iter in range(max_iters):

    xb, yb, lang_ids = get_batch()

    logits, loss = model(xb,lang_ids, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    optimizer.step()

    if iter % eval_interval == 0:
        print(f"Step {iter}: Loss = {loss.item():.4f}")

print(f"🎉 学習完了！最終Loss: {loss.item():.4f}")

In [ ]:
@torch.no_grad()
def generate_word(lang, temperature=0.8, max_len=20):
    if not 0 < temperature < float("inf") or max_len < 1:
        raise ValueError("temperature must be finite and positive; max_len must be positive")
    model.eval()

    # 例: "ar" -> 0
    lang_id = torch.tensor(
        [lang2id[lang]],
        dtype=torch.long
    )

    # <BOS> から開始
    context = torch.tensor(
        [[BOS_ID]],
        dtype=torch.long
    )

    generated = []

    for _ in range(max_len):

        # block_sizeを超えたら後ろだけ使う
        idx_cond = context[:, -block_size:]

        logits, _ = model(
            idx_cond,
            lang_id
        )

        # 最後の位置だけ見る
        logits = logits[:, -1, :] / temperature

        # PADとBOSは生成させない
        logits[:, PAD_ID] = float("-inf")
        logits[:, BOS_ID] = float("-inf")

        probs = F.softmax(logits, dim=-1)

        next_id = torch.multinomial(
            probs,
            num_samples=1
        )

        token_id = next_id.item()

        # finish when EOS appeared
        if token_id == EOS_ID:
            break

        generated.append(id2ipa[token_id])

        context = torch.cat(
            (context, next_id),
            dim=1
        )

    return "".join(generated)
print(generate_word("ar"))
print(generate_word("fi"))
print(generate_word("hu"))
print(generate_word("ru"))

In [ ]:
# mixing libraries
def make_lang_weights(mix):

    weights = torch.zeros(
        1,
        len(lang2id)
    )

    for lang, ratio in mix.items():
        weights[0, lang2id[lang]] = ratio

    if not torch.isfinite(weights).all() or (weights < 0).any() or weights.sum() <= 0:
        raise ValueError("Mixture weights must be finite, nonnegative, and sum to a positive value")
    weights = weights / weights.sum()

    return weights
weights = make_lang_weights({
    "ar": 0.7,
    "fi": 0.3
})

In [ ]:
@torch.no_grad()
def generate_mixed(mix, temperature=0.8, max_len=20):

    if not 0 < temperature < float("inf") or max_len < 1:
        raise ValueError("temperature must be finite and positive; max_len must be positive")
    model.eval()

    lang_weights = make_lang_weights(mix)

    context = torch.tensor(
        [[BOS_ID]],
        dtype=torch.long
    )

    generated = []

    for _ in range(max_len):

        idx_cond = context[:, -block_size:]

        logits, _ = model(
            idx_cond,
            lang_weights=lang_weights
        )

        logits = logits[:, -1, :] / temperature

        logits[:, PAD_ID] = float("-inf")
        logits[:, BOS_ID] = float("-inf")

        probs = F.softmax(logits, dim=-1)

        next_id = torch.multinomial(
            probs,
            num_samples=1
        )

        token_id = next_id.item()

        if token_id == EOS_ID:
            break

        generated.append(id2ipa[token_id])

        context = torch.cat(
            (context, next_id),
            dim=1
        )

    return "".join(generated)
generate_mixed({
    "ar": 0.5,
    "fi": 0.25,
    "ru": 0.25,
})

In [ ]:
for _ in range(30):
    print(generate_mixed({
        "hu": 0.5,
        "fi": 0.25,
        "ru": 0.25,
    }))